# 🏛️ Masterclass 03: Embeddings & Vector Math (The Mathematical Engine of RAG)
### *Google Colab Ready | AI Engineer Interview Preparation (3–5 YOE)*

---

## 📌 Executive Summary
In Retrieval-Augmented Generation (RAG), vector embeddings are the mathematical foundation that bridges human language with fast, scalable nearest-neighbor retrieval.

This notebook centralizes the complete engineering theory, mathematical proofs, and verified interactive Python implementations for:
1. **Mathematical Definition of High-Dimensional Semantic Vectors**
2. **The 3 Core Distance Metrics**: Cosine Similarity, Dot Product, and Euclidean ($L_2$) Distance.
3. **The Unit-Normalization Theorem**: Why $\cos(\theta) = \vec{u} \cdot \vec{v}$ when $\|\vec{u}\| = 1$, enabling SIMD hardware acceleration.
4. **Matryoshka Representation Learning (MRL)**: Compressing 1,536 dimensions down to 512 dimensions for $3\times$ memory & latency savings with $98.5\%$ accuracy.
5. **Asymmetric Retrieval Embeddings**: Solving the query-document syntactic asymmetry problem.
6. **Production Ingestion Engine**: Batching, rate-limiting backoff, and token budgeting.
7. **Senior AI Interview Q&A & Mathematical Derivations**


---
## ⚙️ 1. Environment Setup & Dependency Checks (Colab & Local)
Run the cell below to bootstrap NumPy and verify execution environment.


In [ ]:
import sys
import os
import math
import time
import numpy as np
from typing import List, Dict, Any, Tuple

# Optional packages for full OpenAI API execution if an API key is available
# !pip install numpy openai tiktoken

print('NumPy version:', np.__version__)
print('Environment initialized successfully.')


---
## 📐 2. Mathematical Definition of an Embedding

An embedding model is a parameter-frozen Transformer encoder that maps discrete token sequences to dense real-valued coordinates in continuous $d$-dimensional hyperspace:

$$f: \text{Text String} \longrightarrow \vec{v} \in \mathbb{R}^d$$

- `text-embedding-3-small`: $d = 1,536$
- `text-embedding-3-large`: $d = 3,072$
- `bge-small-en-v1.5`: $d = 384$

Words or sentences conveying similar semantic meaning share similar directional orientations relative to the origin $(0, 0, \dots, 0)$.


In [ ]:
# Simulated 8-dimensional semantic embeddings for three conceptual texts
# Dimensions 0-3 represent Cloud/DevOps semantics; Dimensions 4-7 represent Culinary/Food semantics
doc_cloud_1 = np.array([0.91, 0.85, 0.78, 0.88, 0.05, 0.02, 0.01, 0.04])
doc_cloud_2 = np.array([0.88, 0.82, 0.75, 0.90, 0.04, 0.01, 0.03, 0.02])
doc_cooking = np.array([0.02, 0.05, 0.01, 0.03, 0.89, 0.94, 0.79, 0.85])

print('Simulated Dense Vectors Created:')
print('Doc Cloud 1 (DevOps):', doc_cloud_1)
print('Doc Cloud 2 (DevOps):', doc_cloud_2)
print('Doc Food    (Cooking):', doc_cooking)


---
## 🔬 3. The 3 Core Distance Metrics: Cosine vs Dot Product vs Euclidean ($L_2$)

### A. Dot Product (Inner Product)
$$\vec{u} \cdot \vec{v} = \sum_{i=1}^d u_i v_i = \|\vec{u}\| \|\vec{v}\| \cos(\theta)$$
- **Pros**: Fast computation (single multiply-accumulate loop).
- **Cons**: Sensitive to vector magnitudes. Longer texts with higher norm values score artificially high.

### B. Cosine Similarity (The Industry Standard for Text)
$$\text{Cosine Similarity}(\vec{u}, \vec{v}) = \cos(\theta) = \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \|\vec{v}\|} = \frac{\sum u_i v_i}{\sqrt{\sum u_i^2} \sqrt{\sum v_i^2}}$$
- **Range**: $[-1, 1]$ (text embeddings generally range from $[0, 1]$).
- **Pros**: Measures **pure angular orientation**, completely invariant to text length or magnitude!
$$\text{Cosine Distance} = 1 - \text{Cosine Similarity}$$

### C. Euclidean Distance ($L_2$ Distance)
$$D_{L_2}(\vec{u}, \vec{v}) = \|\vec{u} - \vec{v}\|_2 = \sqrt{\sum_{i=1}^d (u_i - v_i)^2}$$
- **Pros**: Intuitive physical distance in coordinate space.
- **Cons**: Sensitive to vector length differences.


In [ ]:
def compute_dot_product(u: np.ndarray, v: np.ndarray) -> float:
    return float(np.dot(u, v))

def compute_cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return float(np.dot(u, v) / (norm_u * norm_v))

def compute_euclidean_distance(u: np.ndarray, v: np.ndarray) -> float:
    return float(np.linalg.norm(u - v))

# Comparison across pairs
print('=== SIMILARITY / DISTANCE COMPARISON ===')
print('1. Cloud 1 vs Cloud 2 (Same Topic):')
print('   - Cosine Similarity:', round(compute_cosine_similarity(doc_cloud_1, doc_cloud_2), 4))
print('   - Dot Product:      ', round(compute_dot_product(doc_cloud_1, doc_cloud_2), 4))
print('   - Euclidean Distance:', round(compute_euclidean_distance(doc_cloud_1, doc_cloud_2), 4))

print('\n2. Cloud 1 vs Cooking (Different Topics):')
print('   - Cosine Similarity:', round(compute_cosine_similarity(doc_cloud_1, doc_cooking), 4))
print('   - Dot Product:      ', round(compute_dot_product(doc_cloud_1, doc_cooking), 4))
print('   - Euclidean Distance:', round(compute_euclidean_distance(doc_cloud_1, doc_cooking), 4))


---
## ⚡ 4. The Unit-Normalization Theorem (Hardware Acceleration)

### The Production Secret:
Calculating square roots ($\sqrt{\sum x_i^2}$) for millions of vector comparisons during search degrades latency.

If every vector is normalized to **unit length** ($\|\hat{v}\| = 1$) at ingestion time:
$$\hat{v} = \frac{\vec{v}}{\|\vec{v}\|_2} \implies \|\hat{v}\|_2 = 1$$

Then the cosine similarity denominator becomes $1 \times 1 = 1$:
$$\text{Cosine Similarity}(\hat{u}, \hat{v}) = \frac{\hat{u} \cdot \hat{v}}{1 \times 1} = \hat{u} \cdot \hat{v} \quad (\textbf{Raw Dot Product!})$$

And the squared Euclidean distance is directly tied to cosine similarity:
$$\|\hat{u} - \hat{v}\|^2 = \|\hat{u}\|^2 + \|\hat{v}\|^2 - 2(\hat{u} \cdot \hat{v}) = 1 + 1 - 2\cos(\theta) = 2 - 2\cos(\theta)$$

> **Why this matters**:
> Production vector databases (`pgvector`, Pinecone, Milvus) can use hardware-accelerated **AVX-512 SIMD / GPU matrix multiplication (GEMM)** on unit-normalized vectors, computing cosine similarities in nanoseconds with zero division or square roots!


In [ ]:
def l2_normalize(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

# Normalize our test vectors to unit length
norm_cloud_1 = l2_normalize(doc_cloud_1)
norm_cloud_2 = l2_normalize(doc_cloud_2)

print('Magnitude of Normalized Vector 1:', round(float(np.linalg.norm(norm_cloud_1)), 6))
print('Magnitude of Normalized Vector 2:', round(float(np.linalg.norm(norm_cloud_2)), 6))

# Compare Cosine Similarity vs Pure Dot Product on normalized vectors
cos_sim = compute_cosine_similarity(norm_cloud_1, norm_cloud_2)
dot_prod = compute_dot_product(norm_cloud_1, norm_cloud_2)

print('\nVerification of Unit-Normalization Theorem:')
print('Cosine Similarity: ', round(cos_sim, 6))
print('Pure Dot Product:  ', round(dot_prod, 6))
print('Difference (Error):', abs(cos_sim - dot_prod))
assert np.isclose(cos_sim, dot_prod), 'Theorem violation: Dot product must match cosine similarity!'


---
## 🪆 5. Matryoshka Representation Learning (MRL) & Truncation

### The High-Dimensional Memory Crisis:
Storing 10 million chunks using 1,536-dimensional float32 vectors consumes:
$$10,000,000 \times (1536 \times 4\text{ bytes}) \approx 61.44\text{ GB of continuous RAM!}$$

### How MRL Solves This:
Models trained with MRL (like OpenAI `text-embedding-3-small` / `3-large` and `nomic-embed-text`) pack the **most critical semantic features into the leading dimensions** ($d_0$ through $d_{511}$), using trailing dimensions only for fine nuances.

### The Truncation Rule:
When truncating an MRL vector (e.g. from 1,536 to 512 dimensions), you **MUST re-normalize the truncated vector to unit length**:
$$\vec{v}_{512} = \vec{v}[:512]$$
$$\hat{v}_{512} = \frac{\vec{v}_{512}}{\|\vec{v}_{512}\|_2}$$


In [ ]:
# Simulate a 16-dimensional MRL vector where primary semantics are packed in the first 4 dims
np.random.seed(42)
# Primary signal in dims 0-3, lower residual signal in dims 4-15
full_mrl_query = np.concatenate([np.array([0.9, 0.85, 0.7, 0.8]), np.random.normal(0, 0.05, 12)])
full_mrl_match = np.concatenate([np.array([0.88, 0.80, 0.75, 0.82]), np.random.normal(0, 0.05, 12)])
full_mrl_non_match = np.concatenate([np.array([0.1, 0.05, 0.15, 0.02]), np.random.normal(0, 0.05, 12)])

# Normalize full 16-dim vectors
full_mrl_query = l2_normalize(full_mrl_query)
full_mrl_match = l2_normalize(full_mrl_match)
full_mrl_non_match = l2_normalize(full_mrl_non_match)

# Full dimensional similarity
sim_full_match = compute_cosine_similarity(full_mrl_query, full_mrl_match)
sim_full_non_match = compute_cosine_similarity(full_mrl_query, full_mrl_non_match)

# MRL Truncation: Take only first 4 dimensions AND RE-NORMALIZE
trunc_query = l2_normalize(full_mrl_query[:4])
trunc_match = l2_normalize(full_mrl_match[:4])
trunc_non_match = l2_normalize(full_mrl_non_match[:4])

sim_trunc_match = compute_cosine_similarity(trunc_query, trunc_match)
sim_trunc_non_match = compute_cosine_similarity(trunc_query, trunc_non_match)

print('=== MATRYOSHKA DIMENSION TRUNCATION TEST ===')
print('Full 16-dim Match Similarity:     ', round(sim_full_match, 4))
print('Truncated 4-dim Match Similarity:  ', round(sim_trunc_match, 4))
print('Retention Ratio:                   ', round((sim_trunc_match / sim_full_match) * 100, 2), '%')
print('\nNon-Match Discrimination:')
print('Full 16-dim Non-Match Sim:         ', round(sim_full_non_match, 4))
print('Truncated 4-dim Non-Match Sim:     ', round(sim_trunc_non_match, 4))
print('Storage & RAM Savings:              75.0% reduction in vector memory!')


---
## 📦 6. Production Ingestion Engine: Batching & Rate Limiting

### The Bottleneck:
If you ingest 10,000 document chunks using single-item HTTP calls:
1. $10,000$ sequential network handshakes will take over an hour.
2. You will hit API **Rate Limits (HTTP 429 Too Many Requests)**.

### Production Solution:
1. **Batching**: Group chunks into batches of $100 - 2048$ items per call.
2. **Exponential Backoff with Jitter**: Automatically retry with exponential delay when rate-limited.


In [ ]:
class BatchEmbeddingIngestionEngine:
    """
    Simulates production batch ingestion with token budgeting and backoff retry logic.
    """
    def __init__(self, batch_size: int = 4, max_retries: int = 3):
        self.batch_size = batch_size
        self.max_retries = max_retries

    def generate_batch_embeddings(self, texts: List[str]) -> List[np.ndarray]:
        # Deterministic mock embedding generator for simulation
        embeddings = []
        for t in texts:
            seed = sum(ord(c) for c in t) % 1000
            np.random.seed(seed)
            vec = np.random.normal(0, 1, 8)
            embeddings.append(l2_normalize(vec))
        return embeddings

    def ingest_chunks(self, chunks: List[str]) -> List[Dict[str, Any]]:
        results = []
        total_batches = math.ceil(len(chunks) / self.batch_size)
        print(f'Ingesting {len(chunks)} chunks across {total_batches} batches (batch_size={self.batch_size})...')

        for b_idx in range(0, len(chunks), self.batch_size):
            batch = chunks[b_idx:b_idx + self.batch_size]
            retry_count = 0
            while retry_count <= self.max_retries:
                try:
                    # Call API batch endpoint
                    batch_vectors = self.generate_batch_embeddings(batch)
                    for text, vec in zip(batch, batch_vectors):
                        results.append({'text': text, 'embedding_dim': len(vec), 'norm': round(float(np.linalg.norm(vec)), 4)})
                    break
                except Exception as e:
                    retry_count += 1
                    if retry_count > self.max_retries:
                        raise RuntimeError(f'Batch {b_idx} failed after {self.max_retries} retries: {e}')
                    sleep_time = (2 ** retry_count) * 0.1
                    print(f'Transient error. Retrying in {sleep_time:.2f}s...')
                    time.sleep(sleep_time)
        return results

sample_chunks = [
    'Employee health insurance benefits policy.',
    'Quarterly cloud infrastructure budget allocation.',
    'Kubernetes deployment manifest documentation.',
    'Database migration guidelines for PostgreSQL.',
    'API key rotation requirements for compliance.',
    'Data retention schedules for GDPR audits.',
    'Single sign-on OAuth 2.0 configuration guide.'
]

engine = BatchEmbeddingIngestionEngine(batch_size=3)
ingested_records = engine.ingest_chunks(sample_chunks)
print(f'Successfully ingested {len(ingested_records)} chunks in batch mode.')
print('Sample Ingested Record:', ingested_records[0])


---
## 📊 7. Distance Metrics & Model Comparison Matrix

| Metric / Feature | Formula | Magnitude Invariance | Best For | Hardware Acceleration |
| :--- | :--- | :--- | :--- | :--- |
| **Cosine Similarity** | $\frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \|\vec{v}\|}$ | **Yes (100% invariant)** | Text retrieval & semantic search | Slower unless vectors pre-normalized |
| **Dot Product** | $\sum u_i v_i$ | No (favors longer vectors) | Pre-normalized unit vectors | **Blazing fast (SIMD GEMM / AVX-512)** |
| **Euclidean ($L_2$)** | $\sqrt{\sum (u_i - v_i)^2}$ | No (penalizes magnitude) | Image features & normalized vectors | Fast on fixed-norm vectors |

### Model Benchmark Reference:
| Model Name | Dimensions | Max Tokens | MRL Support | MTEB Retrieval Score |
| :--- | :--- | :--- | :--- | :--- |
| **OpenAI `text-embedding-3-small`** | 1,536 (or 512) | 8,191 | **Yes** | 62.3% |
| **OpenAI `text-embedding-3-large`** | 3,072 (or 1024) | 8,191 | **Yes** | 64.6% |
| **BAAI `bge-small-en-v1.5`** | 384 | 512 | No | 62.1% |
| **Nomic `nomic-embed-text-v1.5`** | 768 (or 256) | 8,192 | **Yes** | 62.2% |


---
## 🎯 8. Senior AI Interview Questions: Embeddings & Vector Math

### Q1: Why is Dot Product mathematically identical to Cosine Similarity for unit vectors?
> **Answer**: Cosine similarity is defined as $\frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \|\vec{v}\|}$. When vectors are $L_2$-normalized to unit length, $\|\vec{u}\| = 1$ and $\|\vec{v}\| = 1$. The denominator becomes $1 \times 1 = 1$, simplifying the entire formula to $\vec{u} \cdot \vec{v}$. This allows vector databases to execute pure dot products via SIMD hardware instructions instead of calculating square roots.

---

### Q2: What is Matryoshka Representation Learning (MRL), and what step must you perform after truncating?
> **Answer**: MRL trains embedding models with nested loss functions across varying sub-dimensions, forcing the network to pack the highest-density semantic signals into the earliest dimensions (e.g., the first 512 of 1,536). When slicing an MRL vector (`vec[:512]`), **you must immediately re-normalize the truncated vector to unit length ($L_2$ norm = 1)**, otherwise cosine similarity and dot product calculations will be distorted.

---

### Q3: When should you choose Cosine Similarity over Euclidean Distance for text RAG?
> **Answer**: In text documents, two chunks discussing the exact same concept may vary in length (e.g. a 3-line summary vs an 8-line detailed paragraph). Cosine similarity measures the directional angle between vectors and is invariant to length, whereas Euclidean distance penalizes magnitude discrepancies. Therefore, Cosine Similarity (or normalized Dot Product) is the standard for text retrieval.
